# Qiskit Fall Fest 2026 — PS-O2

**Scaffold only:** no implementation or benchmark results are claimed yet.

Follow `TEAM_IMPLEMENTATION_PLAN.md`, `AI_WORKING_RULES.md`, and the team-approved `INTERFACES.md`. The common official risk matrix is pending.


## 1. Environment and reproducibility

Record tested Python/package versions after validating a clean runtime.


In [1]:
import sys
print('Python:', sys.version)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [2]:

import numpy as np
import itertools

print("NumPy version:", np.__version__)
print("Standard library itertools: imported successfully")


NumPy version: 2.1.3
Standard library itertools: imported successfully


## 2. Problem inputs

Load the approved configuration from the shared module. Clearly label custom-matrix experiments.


In [3]:

# Problem parameters provided by the organizers

NUM_ASSETS = 4
RETURNS = np.array([5.0, 3.0, 4.0, 2.0])

RISK_AVERSION = 2.0
BUDGET_PENALTY = 20.0
TARGET_ASSETS = 2
QAOA_P = 1

print("Assets:", NUM_ASSETS)
print("Returns:", RETURNS)
print("Risk aversion:", RISK_AVERSION)
print("Budget penalty:", BUDGET_PENALTY)
print("Required assets:", TARGET_ASSETS)
print("QAOA depth p:", QAOA_P)


Assets: 4
Returns: [5. 3. 4. 2.]
Risk aversion: 2.0
Budget penalty: 20.0
Required assets: 2
QAOA depth p: 1


In [4]:
# Official risk matrix is pending from the organizers
SIGMA = None

def validate_risk_matrix(sigma):
    if sigma is None:
        raise ValueError("The official risk matrix has not been provided.")

    sigma = np.asarray(sigma, dtype=float)

    if sigma.shape != (NUM_ASSETS, NUM_ASSETS):
        raise ValueError("Risk matrix must be 4x4.")

    if not np.all(np.isfinite(sigma)):
        raise ValueError("Risk matrix must contain finite values.")

    if not np.allclose(sigma, sigma.T):
        raise ValueError("Risk matrix must be symmetric.")

    return sigma

print("Risk matrix input prepared.")
print("Official matrix available:", SIGMA is not None)

Risk matrix input prepared.
Official matrix available: False


In [5]:

def portfolio_objective(x, returns=RETURNS, sigma=SIGMA):
    """
    Evaluate the organizer-defined PS-O2 maximization objective.

    x: binary vector indicating which assets are selected.
    returns: expected returns for the four assets.
    sigma: 4x4 risk matrix.
    """
    x = np.asarray(x, dtype=int)

    if x.shape != (NUM_ASSETS,) or not np.all(np.isin(x, [0, 1])):
        raise ValueError("x must contain exactly four binary values.")

    returns = np.asarray(returns, dtype=float)
    if returns.shape != (NUM_ASSETS,):
        raise ValueError("Returns must contain four values.")

    sigma = validate_risk_matrix(sigma)

    return float(
        returns @ x
        - RISK_AVERSION * (x @ sigma @ x)
        - BUDGET_PENALTY * (np.sum(x) - TARGET_ASSETS) ** 2
    )

print("Portfolio objective function defined.")


Portfolio objective function defined.


In [6]:

print("Objective function ready:", callable(portfolio_objective))


Objective function ready: True


## 3. Mathematical formulation and QUBO verification

Document the objective, penalty, sign convention, constant offset, and checks over all 16 bitstrings.


In [7]:

def build_qubo(returns, sigma, risk_aversion=RISK_AVERSION,
               penalty=BUDGET_PENALTY, target=TARGET_ASSETS):
    """
    Build C(x) = -f(x) as:
        C(x) = offset + sum_i Q[i,i]*x_i
                         + sum_{i<j} Q[i,j]*x_i*x_j

    Upper-triangular convention: each pair coefficient is stored once.
    """
    returns = np.asarray(returns, dtype=float)
    sigma = validate_risk_matrix(sigma)

    n = len(returns)
    Q = np.zeros((n, n), dtype=float)

    # Negative return and negative risk-adjusted reward
    for i in range(n):
        Q[i, i] += -returns[i] + risk_aversion * sigma[i, i]

    for i in range(n):
        for j in range(i + 1, n):
            Q[i, j] += 2 * risk_aversion * sigma[i, j]

    # Expand penalty * (sum(x) - target)^2, using x_i^2 = x_i
    for i in range(n):
        Q[i, i] += penalty * (1 - 2 * target)

    for i in range(n):
        for j in range(i + 1, n):
            Q[i, j] += 2 * penalty

    offset = penalty * target**2
    return Q, offset


print("QUBO builder defined.")


QUBO builder defined.


In [8]:

print("QUBO builder ready:", callable(build_qubo))


QUBO builder ready: True


In [9]:

# TEMPORARY TEST DATA ONLY — NOT OFFICIAL BENCHMARK DATA
SIGMA_TEST = np.array([
    [0.40, 0.05, 0.02, 0.01],
    [0.05, 0.30, 0.04, 0.02],
    [0.02, 0.04, 0.35, 0.03],
    [0.01, 0.02, 0.03, 0.25]
])

# Build QUBO using the temporary matrix
Q_test, offset_test = build_qubo(RETURNS, SIGMA_TEST)

def qubo_cost(x, Q, offset=0.0):
    """Evaluate upper-triangular QUBO cost."""
    x = np.asarray(x, dtype=float)
    return float(offset + sum(
        Q[i, j] * x[i] * x[j]
        for i in range(len(x))
        for j in range(i, len(x))
    ))

# Check every possible binary portfolio
max_error = 0.0

for bits in itertools.product([0, 1], repeat=NUM_ASSETS):
    x = np.array(bits, dtype=int)

    original_maximization = (
        RETURNS @ x
        - RISK_AVERSION * (x @ SIGMA_TEST @ x)
        - BUDGET_PENALTY * (sum(x) - TARGET_ASSETS) ** 2
    )

    converted_minimization = qubo_cost(x, Q_test, offset_test)

    error = abs(converted_minimization + original_maximization)
    max_error = max(max_error, error)

print("Test matrix only — not official results")
print("Portfolios checked:", 2 ** NUM_ASSETS)
print("Maximum conversion error:", max_error)
print("QUBO conversion correct:", np.isclose(max_error, 0.0))


Test matrix only — not official results
Portfolios checked: 16
Maximum conversion error: 7.993605777301127e-15
QUBO conversion correct: True


## 4. Classical reference solution

Report the exact best feasible portfolio for the active configuration.


In [10]:

def classical_baseline(returns, sigma):
    """Exhaustively evaluate all 16 binary portfolios."""
    returns = np.asarray(returns, dtype=float)
    sigma = validate_risk_matrix(sigma)

    results = []

    for bits in itertools.product([0, 1], repeat=NUM_ASSETS):
        x = np.array(bits, dtype=int)

        reward = float(returns @ x)
        risk = float(x @ sigma @ x)
        selected_count = int(np.sum(x))

        objective = (
            reward
            - RISK_AVERSION * risk
            - BUDGET_PENALTY * (selected_count - TARGET_ASSETS) ** 2
        )

        results.append({
            "bits": x.tolist(),
            "selected_assets": np.flatnonzero(x).tolist(),
            "selected_count": selected_count,
            "return": reward,
            "risk": risk,
            "objective": float(objective),
            "feasible": selected_count == TARGET_ASSETS
        })

    best_feasible = max(
        (row for row in results if row["feasible"]),
        key=lambda row: row["objective"]
    )

    best_penalized = max(results, key=lambda row: row["objective"])

    return results, best_feasible, best_penalized


print("Classical baseline function defined:", callable(classical_baseline))


Classical baseline function defined: True


In [12]:

def show_baseline_result(label, result):
    print(label)
    print("Selected assets:", result["selected_assets"])
    print("Binary vector:", result["bits"])
    print("Return:", result["return"])
    print("Risk:", result["risk"])
    print("Penalized objective:", result["objective"])
    print("Constraint satisfied:", result["feasible"])


print("Result-display function ready.")


Result-display function ready.


## 5. QAOA at p=1

Record settings, measurement counts, decoded samples and validity metrics.


In [13]:

import importlib.util
import sys

print("Python:", sys.version.split()[0])

for package in ["qiskit", "qiskit_aer", "scipy"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package} installed:", available)

if importlib.util.find_spec("qiskit") is not None:
    import qiskit
    print("Qiskit version:", qiskit.__version__)


Python: 3.13.16
qiskit installed: False
qiskit_aer installed: False
scipy installed: True


In [14]:

%pip install -q qiskit qiskit-aer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.3 MB/s eta 0:00:00


In [15]:

import qiskit
import qiskit_aer

print("Qiskit version:", qiskit.__version__)
print("Qiskit Aer version:", qiskit_aer.__version__)
print("Quantum simulation packages are ready!")


Qiskit version: 2.5.2
Qiskit Aer version: 0.17.2
Quantum simulation packages are ready!


In [16]:

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

NUM_QUBITS = NUM_ASSETS
gamma = Parameter("gamma")
beta = Parameter("beta")

qaoa_circuit = QuantumCircuit(NUM_QUBITS)

# Initial equal superposition
qaoa_circuit.h(range(NUM_QUBITS))

# Placeholder cost-layer structure:
# RZ rotations act on individual qubits.
# The verified QUBO coefficients will determine their angles.
for i in range(NUM_QUBITS):
    qaoa_circuit.rz(gamma, i)

# Standard X-mixer layer
for i in range(NUM_QUBITS):
    qaoa_circuit.rx(2 * beta, i)

qaoa_circuit.measure_all()

print("Qubits:", qaoa_circuit.num_qubits)
print("QAOA layers:", QAOA_P)
print("Parameters:", qaoa_circuit.parameters)
print("Circuit depth:", qaoa_circuit.depth())


Qubits: 4
QAOA layers: 1
Parameters: ParameterView([Parameter(beta), Parameter(gamma)])
Circuit depth: 4


In [17]:

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter


def build_qaoa_circuit(Q):
    """
    Build a p=1 QAOA circuit for an upper-triangular QUBO:
    C(x) = sum_i Q[i,i]*x_i + sum_{i<j} Q[i,j]*x_i*x_j

    Constant offsets are omitted because they only add a
    global phase and do not affect measurement probabilities.
    """
    Q = np.asarray(Q, dtype=float)
    n = Q.shape[0]

    if Q.shape != (NUM_ASSETS, NUM_ASSETS):
        raise ValueError("Q must be a 4x4 matrix.")

    if not np.all(np.isfinite(Q)):
        raise ValueError("Q must contain finite values.")

    gamma = Parameter("gamma")
    beta = Parameter("beta")

    circuit = QuantumCircuit(n)

    # Start in an equal superposition
    circuit.h(range(n))

    # Convert QUBO coefficients to Ising Z and ZZ coefficients.
    z_coeff = np.zeros(n, dtype=float)

    for i in range(n):
        z_coeff[i] = -Q[i, i] / 2

    for i in range(n):
        for j in range(i + 1, n):
            pair_coeff = Q[i, j]

            z_coeff[i] -= pair_coeff / 4
            z_coeff[j] -= pair_coeff / 4

            # ZZ interaction for Q[i,j] * x_i * x_j
            if not np.isclose(pair_coeff, 0.0):
                circuit.cx(i, j)
                circuit.rz(gamma * pair_coeff / 2, j)
                circuit.cx(i, j)

    # Single-qubit Z terms
    for i in range(n):
        if not np.isclose(z_coeff[i], 0.0):
            circuit.rz(2 * gamma * z_coeff[i], i)

    # Standard QAOA X-mixer
    for i in range(n):
        circuit.rx(2 * beta, i)

    circuit.measure_all()

    return circuit


print("QUBO-based QAOA circuit builder defined.")


QUBO-based QAOA circuit builder defined.


In [18]:

Q_test, offset_test = build_qubo(RETURNS, SIGMA_TEST)

qaoa_test_circuit = build_qaoa_circuit(Q_test)

print("Qubits:", qaoa_test_circuit.num_qubits)
print("Parameters:", qaoa_test_circuit.parameters)
print("Circuit depth:", qaoa_test_circuit.depth())
print("Circuit operations:", qaoa_test_circuit.count_ops())

qaoa_test_circuit.draw("text")


Qubits: 4
Parameters: ParameterView([Parameter(beta), Parameter(gamma)])
Circuit depth: 19
Circuit operations: OrderedDict({'cx': 12, 'rz': 10, 'h': 4, 'rx': 4, 'measure': 4, 'barrier': 1})


┌───┐                                                              »
   q_0: ┤ H ├──■──────────────────────■────■───────────────────────■────■──»
        ├───┤┌─┴─┐┌────────────────┐┌─┴─┐  │                       │    │  »
   q_1: ┤ H ├┤ X ├┤ Rz(20.1*gamma) ├┤ X ├──┼───────────────────────┼────┼──»
        ├───┤└───┘└────────────────┘└───┘┌─┴─┐┌─────────────────┐┌─┴─┐  │  »
   q_2: ┤ H ├────────────────────────────┤ X ├┤ Rz(20.04*gamma) ├┤ X ├──┼──»
        ├───┤                            └───┘└─────────────────┘└───┘┌─┴─┐»
   q_3: ┤ H ├─────────────────────────────────────────────────────────┤ X ├»
        └───┘                                                         └───┘»
meas: 4/═══════════════════════════════════════════════════════════════════»
                                                                           »
«                                                   »
«   q_0: ────────────────────────────────────────■──»
«                                                │  »
«   q_1: ─────────■──────────────────────────────┼──»
«               ┌─┴─┐       ┌─────────────────┐  │  »
«   q_2: ───────┤ X ├───────┤ Rz(20.08*gamma) ├──┼──»
«        ┌──────┴───┴──────┐└─────────────────┘┌─┴─┐»
«   q_3: ┤ Rz(20.02*gamma) ├───────────────────┤ X ├»
«        └─────────────────┘                   └───┘»
«meas: 4/═══════════════════════════════════════════»
«                                                   »
«        ┌─────────────────────────────┐┌────────────┐                        »
«   q_0: ┤ Rz(4.040000000000003*gamma) ├┤ Rx(2*beta) ├────────────────────────»
«        └─────────────────────────────┘└────────────┘                        »
«   q_1: ───────────────■─────────────────────■────────────────────────────■──»
«                     ┌─┴─┐                   │                            │  »
«   q_2: ─────────────┤ X ├───────────────────┼────────────────────────────┼──»
«                     └───┘                 ┌─┴─┐     ┌─────────────────┐┌─┴─┐»
«   q_3: ───────────────────────────────────┤ X ├─────┤ Rz(20.04*gamma) ├┤ X ├»
«                                           └───┘     └─────────────────┘└───┘»
«meas: 4/═════════════════════════════════════════════════════════════════════»
«                                                                             »
«                                                                »
«   q_0: ────────────────────────────────────────────────────────»
«        ┌──────────────────────────────┐   ┌────────────┐       »
«   q_1: ┤ Rz(2.1799999999999997*gamma) ├───┤ Rx(2*beta) ├───────»
«        └──────────────────────────────┘   └────────────┘       »
«   q_2: ───────────────■─────────────────────────────────────■──»
«                     ┌─┴─┐              ┌─────────────────┐┌─┴─┐»
«   q_3: ─────────────┤ X ├──────────────┤ Rz(20.06*gamma) ├┤ X ├»
«                     └───┘              └─────────────────┘└───┘»
«meas: 4/════════════════════════════════════════════════════════»
«                                                                »
«                                                      ░ ┌─┐         
«   q_0: ──────────────────────────────────────────────░─┤M├─────────
«                                                      ░ └╥┘┌─┐      
«   q_1: ──────────────────────────────────────────────░──╫─┤M├──────
«        ┌─────────────────────────────┐┌────────────┐ ░  ║ └╥┘┌─┐   
«   q_2: ┤ Rz(3.120000000000001*gamma) ├┤ Rx(2*beta) ├─░──╫──╫─┤M├───
«        ├─────────────────────────────┤├────────────┤ ░  ║  ║ └╥┘┌─┐
«   q_3: ┤ Rz(1.380000000000006*gamma) ├┤ Rx(2*beta) ├─░──╫──╫──╫─┤M├
«        └─────────────────────────────┘└────────────┘ ░  ║  ║  ║ └╥┘
«meas: 4/═════════════════════════════════════════════════╩══╩══╩══╩═
«                                                         0  1  2  3

In [19]:

from qiskit.quantum_info import Statevector
import numpy as np

def expected_qubo_cost(circuit, Q, offset=0.0, gamma_value=0.5, beta_value=0.5):
    """
    Calculate the expected QUBO cost using an ideal statevector.
    Lower cost is better because QUBO minimizes -objective.
    """
    # Remove measurements before statevector simulation
    unitary_circuit = circuit.remove_final_measurements(inplace=False)

    # Bind the QAOA parameters
    parameters = {p.name: p for p in unitary_circuit.parameters}
    bound_circuit = unitary_circuit.assign_parameters({
        parameters["gamma"]: gamma_value,
        parameters["beta"]: beta_value
    })

    # Simulate the quantum state
    state = Statevector.from_instruction(bound_circuit)
    probabilities = state.probabilities()

    # Evaluate every computational-basis bitstring
    expected_cost = 0.0

    for index, probability in enumerate(probabilities):
        # Qiskit stores qubit 0 in the least-significant bit
        x = np.array([(index >> i) & 1 for i in range(NUM_ASSETS)])
        cost = float(x @ Q @ x + offset)
        expected_cost += probability * cost

    return float(expected_cost)

test_cost = expected_qubo_cost(
    qaoa_test_circuit,
    Q_test,
    offset=offset_test,
    gamma_value=0.5,
    beta_value=0.5
)

print("Test expected QUBO cost:", test_cost)
print("Simulation completed successfully.")


Test expected QUBO cost: 24.878571636535973
Simulation completed successfully.


In [20]:

from scipy.optimize import minimize

def qaoa_objective(params, circuit, Q, offset):
    gamma_value, beta_value = params

    return expected_qubo_cost(
        circuit,
        Q,
        offset=offset,
        gamma_value=gamma_value,
        beta_value=beta_value
    )

initial_parameters = [0.5, 0.5]

optimization_result = minimize(
    qaoa_objective,
    x0=initial_parameters,
    args=(qaoa_test_circuit, Q_test, offset_test),
    method="COBYLA",
    options={
        "maxiter": 100,
        "rhobeg": 0.5,
        "tol": 1e-4
    }
)

best_gamma, best_beta = optimization_result.x

print("Optimization successful:", optimization_result.success)
print("Optimizer message:", optimization_result.message)
print("Best gamma:", best_gamma)
print("Best beta:", best_beta)
print("Best expected QUBO cost:", optimization_result.fun)
print("Initial expected QUBO cost:", test_cost)


Optimization successful: False
Optimizer message: Return from COBYLA because the objective function has been evaluated MAXFUN times.
Best gamma: 1.1755676977281562
Best beta: 0.7283396165139331
Best expected QUBO cost: 1.870832267303496
Initial expected QUBO cost: 24.878571636535973


In [21]:

optimization_result_2 = minimize(
    qaoa_objective,
    x0=optimization_result.x,
    args=(qaoa_test_circuit, Q_test, offset_test),
    method="COBYLA",
    options={
        "maxiter": 500,
        "rhobeg": 0.1,
        "tol": 1e-5
    }
)

best_gamma, best_beta = optimization_result_2.x

print("Optimization successful:", optimization_result_2.success)
print("Optimizer message:", optimization_result_2.message)
print("Best gamma:", best_gamma)
print("Best beta:", best_beta)
print("Best expected QUBO cost:", optimization_result_2.fun)
print("Previous best expected QUBO cost:", optimization_result.fun)


Optimization successful: False
Optimizer message: Return from COBYLA because the objective function has been evaluated MAXFUN times.
Best gamma: 1.1753022772507937
Best beta: 0.7887492867081691
Best expected QUBO cost: 1.7014347063758097
Previous best expected QUBO cost: 1.870832267303496


In [22]:

from qiskit.quantum_info import Statevector

# Use the best parameters found in the latest optimization
best_gamma, best_beta = optimization_result_2.x

unitary_circuit = qaoa_test_circuit.remove_final_measurements(inplace=False)
parameters = {p.name: p for p in unitary_circuit.parameters}

bound_circuit = unitary_circuit.assign_parameters({
    parameters["gamma"]: best_gamma,
    parameters["beta"]: best_beta
})

state = Statevector.from_instruction(bound_circuit)
probabilities = state.probabilities()

# List all possible portfolios, ordered by probability
portfolio_results = []

for index, probability in enumerate(probabilities):
    x = np.array([(index >> i) & 1 for i in range(NUM_ASSETS)])
    objective_value = portfolio_objective(x, sigma=SIGMA_TEST)

    portfolio_results.append({
        "bitstring": "".join(map(str, x[::-1])),
        "assets_selected": np.flatnonzero(x).tolist(),
        "selected_count": int(np.sum(x)),
        "original_objective": objective_value,
        "probability": float(probability)
    })

portfolio_results.sort(key=lambda row: row["probability"], reverse=True)

print("Top 5 portfolios from optimized QAOA:")
for row in portfolio_results[:5]:
    print(row)

# Classical exhaustive search using the same temporary matrix
all_classical, best_feasible, best_penalized = classical_baseline(
    RETURNS, SIGMA_TEST
)

print("\nBest classical feasible portfolio:")
print(best_feasible)

print("\nBest classical penalized portfolio:")
print(best_penalized)


Top 5 portfolios from optimized QAOA:
{'bitstring': '0101', 'assets_selected': [0, 2], 'selected_count': 2, 'original_objective': 7.42, 'probability': 0.29231294762166915}
{'bitstring': '1010', 'assets_selected': [1, 3], 'selected_count': 2, 'original_objective': 3.82, 'probability': 0.28712244008531235}
{'bitstring': '1110', 'assets_selected': [1, 2, 3], 'selected_count': 3, 'original_objective': -13.16, 'probability': 0.0988276562383713}
{'bitstring': '0001', 'assets_selected': [0], 'selected_count': 1, 'original_objective': -15.8, 'probability': 0.09356839621554633}
{'bitstring': '1000', 'assets_selected': [3], 'selected_count': 1, 'original_objective': -18.5, 'probability': 0.08764162601367419}

Best classical feasible portfolio:
{'bits': [1, 0, 1, 0], 'selected_assets': [0, 2], 'selected_count': 2, 'return': 9.0, 'risk': 0.79, 'objective': 7.42, 'feasible': True}

Best classical penalized portfolio:
{'bits': [1, 0, 1, 0], 'selected_assets': [0, 2], 'selected_count': 2, 'return': 9

In [23]:

# Evaluate constraint satisfaction across the complete QAOA state
valid_probability = sum(
    row["probability"]
    for row in portfolio_results
    if row["selected_count"] == TARGET_ASSETS
)

optimal_bits = tuple(best_feasible["bits"])

optimal_probability = sum(
    row["probability"]
    for row in portfolio_results
    if tuple(
        int(bit) for bit in row["bitstring"][::-1]
    ) == optimal_bits
)

print(f"Probability of satisfying the constraint: {valid_probability:.4f}")
print(f"Probability of measuring the classical optimum: {optimal_probability:.4f}")
print(f"Constraint violation probability: {1 - valid_probability:.4f}")
print(f"Optimal portfolio objective: {best_feasible['objective']:.4f}")


Probability of satisfying the constraint: 0.6368
Probability of measuring the classical optimum: 0.2923
Constraint violation probability: 0.3632
Optimal portfolio objective: 7.4200


## 6. Processor A/B benchmark

Benchmark the same agreed logical circuit and report measured CX count, depth and SWAP overhead.


In [24]:

# Processor connectivity from the hackathon problem statement
PROCESSOR_A_EDGES = [
    (0, 1), (1, 2), (1, 3), (3, 4)
]

PROCESSOR_B_EDGES = [
    (0, 1), (1, 2), (1, 6), (2, 3),
    (2, 6), (2, 5), (3, 4), (4, 5),
    (5, 6), (6, 7), (6, 8), (8, 0)
]

print("Processor A:", len(PROCESSOR_A_EDGES), "connectivity edges")
print("Processor B:", len(PROCESSOR_B_EDGES), "connectivity edges")


Processor A: 4 connectivity edges
Processor B: 12 connectivity edges


In [25]:

from qiskit import transpile
from qiskit.transpiler import CouplingMap

# Build coupling maps from the specified processor topologies
coupling_map_A = CouplingMap(
    couplinglist=PROCESSOR_A_EDGES
)

coupling_map_B = CouplingMap(
    couplinglist=PROCESSOR_B_EDGES
)

# Transpile the same logical QAOA circuit for each processor
transpiled_A = transpile(
    qaoa_test_circuit,
    coupling_map=coupling_map_A,
    basis_gates=["rz", "sx", "x", "cx"],
    optimization_level=0,
    seed_transpiler=42
)

transpiled_B = transpile(
    qaoa_test_circuit,
    coupling_map=coupling_map_B,
    basis_gates=["rz", "sx", "x", "cx"],
    optimization_level=0,
    seed_transpiler=42
)

print("Processor A")
print("Physical qubits:", transpiled_A.num_qubits)
print("Depth:", transpiled_A.depth())
print("Operations:", transpiled_A.count_ops())

print("\nProcessor B")
print("Physical qubits:", transpiled_B.num_qubits)
print("Depth:", transpiled_B.depth())
print("Operations:", transpiled_B.count_ops())


Processor A
Physical qubits: 5
Depth: 91
Operations: OrderedDict({'rz': 86, 'sx': 40, 'cx': 21, 'measure': 4, 'barrier': 1})

Processor B
Physical qubits: 9
Depth: 85
Operations: OrderedDict({'rz': 102, 'sx': 48, 'cx': 24, 'measure': 4, 'barrier': 1})


In [26]:

logical_cx_count = qaoa_test_circuit.count_ops().get("cx", 0)

def calculate_gate_metrics(circuit, logical_cx_count):
    ops = circuit.count_ops()
    physical_cx = ops.get("cx", 0)

    return {
        "depth": circuit.depth(),
        "physical_cx_count": physical_cx,
        "logical_cx_count": logical_cx_count,
        "additional_cx_gates": physical_cx - logical_cx_count,
        "cx_count_increase_percent": (
            (physical_cx - logical_cx_count) / logical_cx_count * 100
            if logical_cx_count else 0.0
        ),
        "explicit_swap_count": ops.get("swap", 0)
    }

metrics_A = calculate_gate_metrics(transpiled_A, logical_cx_count)
metrics_B = calculate_gate_metrics(transpiled_B, logical_cx_count)

print("Logical CNOT count:", logical_cx_count)
print("\nProcessor A metrics:")
print(metrics_A)
print("\nProcessor B metrics:")
print(metrics_B)


Logical CNOT count: 12

Processor A metrics:
{'depth': 91, 'physical_cx_count': 21, 'logical_cx_count': 12, 'additional_cx_gates': 9, 'cx_count_increase_percent': 75.0, 'explicit_swap_count': 0}

Processor B metrics:
{'depth': 85, 'physical_cx_count': 24, 'logical_cx_count': 12, 'additional_cx_gates': 12, 'cx_count_increase_percent': 100.0, 'explicit_swap_count': 0}


In [27]:

def inspect_transpiled_circuit(name, circuit):
    print(f"\n{name}")
    print("-" * 35)
    print("Operations:", dict(circuit.count_ops()))
    print("Depth:", circuit.depth())
    print("Two-qubit operations:")

    for instruction, count in circuit.count_ops().items():
        if circuit.find_bit if False else False:
            pass
        if instruction in ("cx", "cz", "ecr", "swap"):
            print(f"  {instruction}: {count}")

    print("\nCircuit layout:", circuit.layout)

inspect_transpiled_circuit("Processor A", transpiled_A)
inspect_transpiled_circuit("Processor B", transpiled_B)



Processor A
-----------------------------------
Operations: {'rz': 86, 'sx': 40, 'cx': 21, 'measure': 4, 'barrier': 1}
Depth: 91
Two-qubit operations:
  cx: 21

Circuit layout: TranspileLayout(initial_layout=Layout({
0: <Qubit register=(4, "q"), index=0>,
1: <Qubit register=(4, "q"), index=1>,
2: <Qubit register=(4, "q"), index=2>,
3: <Qubit register=(4, "q"), index=3>,
4: <Qubit register=(1, "ancilla"), index=0>
}), input_qubit_mapping={<Qubit register=(4, "q"), index=0>: 0, <Qubit register=(4, "q"), index=1>: 1, <Qubit register=(4, "q"), index=2>: 2, <Qubit register=(4, "q"), index=3>: 3, <Qubit register=(1, "ancilla"), index=0>: 4}, final_layout=Layout({
0: <Qubit register=(5, "q"), index=0>,
3: <Qubit register=(5, "q"), index=1>,
2: <Qubit register=(5, "q"), index=2>,
1: <Qubit register=(5, "q"), index=3>,
4: <Qubit register=(5, "q"), index=4>
}), _input_qubit_count=4, _output_qubit_list=[<Qubit register=(5, "q"), index=0>, <Qubit register=(5, "q"), index=1>, <Qubit register=(5, "

In [28]:

from qiskit.transpiler import generate_preset_pass_manager

def inspect_routing_stage(circuit, coupling_map, name):
    pm = generate_preset_pass_manager(
        optimization_level=0,
        coupling_map=coupling_map,
        basis_gates=["rz", "sx", "x", "cx"],
        seed_transpiler=42
    )

    stages = pm.run(circuit, callback=None)

    print(f"{name}: full transpilation complete")
    print("Final depth:", stages.depth())
    print("Final operations:", dict(stages.count_ops()))

    return stages

# Re-run using the same settings for reproducibility
transpiled_A_check = inspect_routing_stage(
    qaoa_test_circuit, coupling_map_A, "Processor A"
)

transpiled_B_check = inspect_routing_stage(
    qaoa_test_circuit, coupling_map_B, "Processor B"
)


Processor A: full transpilation complete
Final depth: 91
Final operations: {'rz': 86, 'sx': 40, 'cx': 21, 'measure': 4, 'barrier': 1}
Processor B: full transpilation complete
Final depth: 85
Final operations: {'rz': 102, 'sx': 48, 'cx': 24, 'measure': 4, 'barrier': 1}


In [29]:

import pandas as pd

benchmark_summary = pd.DataFrame([
    {
        "Processor": "A",
        "Physical qubits": transpiled_A.num_qubits,
        "Circuit depth": transpiled_A.depth(),
        "CNOT gates": transpiled_A.count_ops().get("cx", 0),
        "RZ gates": transpiled_A.count_ops().get("rz", 0),
        "SX gates": transpiled_A.count_ops().get("sx", 0),
        "Explicit SWAP gates": transpiled_A.count_ops().get("swap", 0),
        "Noise model": "None (ideal topology-only)"
    },
    {
        "Processor": "B",
        "Physical qubits": transpiled_B.num_qubits,
        "Circuit depth": transpiled_B.depth(),
        "CNOT gates": transpiled_B.count_ops().get("cx", 0),
        "RZ gates": transpiled_B.count_ops().get("rz", 0),
        "SX gates": transpiled_B.count_ops().get("sx", 0),
        "Explicit SWAP gates": transpiled_B.count_ops().get("swap", 0),
        "Noise model": "None (ideal topology-only)"
    }
])

display(benchmark_summary)

print("\nImportant limitations:")
print("- Uses SIGMA_TEST, not the official risk matrix.")
print("- Uses optimization_level=0 and transpiler seed 42.")
print("- Additional CNOT gates are not an isolated routing-overhead measurement.")
print("- No hardware noise or device error rates are simulated.")


,Processor,Physical qubits,Circuit depth,CNOT gates,RZ gates,SX gates,Explicit SWAP gates,Noise model
0,A,5,91,21,86,40,0,None (ideal topology-only)
1,B,9,85,24,102,48,0,None (ideal topology-only)



Important limitations:
- Uses SIGMA_TEST, not the official risk matrix.
- Uses optimization_level=0 and transpiler seed 42.
- Additional CNOT gates are not an isolated routing-overhead measurement.
- No hardware noise or device error rates are simulated.


In [31]:

from qiskit.transpiler import generate_preset_pass_manager

def routing_stage_report(circuit, coupling_map, processor_name):
    # Build the complete preset pass manager
    pm = generate_preset_pass_manager(
        optimization_level=0,
        coupling_map=coupling_map,
        basis_gates=["rz", "sx", "x", "cx"],
        seed_transpiler=42
    )

    # Run the full pipeline to obtain a correctly mapped circuit
    # This validates the topology and provides final gate metrics.
    transpiled = pm.run(circuit)

    print(f"\n{processor_name}")
    print("-" * 35)
    print("Physical qubits:", transpiled.num_qubits)
    print("Circuit depth:", transpiled.depth())
    print("Operations:", dict(transpiled.count_ops()))
    print("Explicit SWAP gates:", transpiled.count_ops().get("swap", 0))
    print("CNOT gates:", transpiled.count_ops().get("cx", 0))

    return transpiled

transpiled_A_check = routing_stage_report(
    qaoa_test_circuit, coupling_map_A, "Processor A"
)

transpiled_B_check = routing_stage_report(
    qaoa_test_circuit, coupling_map_B, "Processor B"
)



Processor A
-----------------------------------
Physical qubits: 5
Circuit depth: 91
Operations: {'rz': 86, 'sx': 40, 'cx': 21, 'measure': 4, 'barrier': 1}
Explicit SWAP gates: 0
CNOT gates: 21

Processor B
-----------------------------------
Physical qubits: 9
Circuit depth: 85
Operations: {'rz': 102, 'sx': 48, 'cx': 24, 'measure': 4, 'barrier': 1}
Explicit SWAP gates: 0
CNOT gates: 24


In [32]:

# Recheck the QUBO representation against the original objective
max_error = 0.0

for bits in itertools.product([0, 1], repeat=NUM_ASSETS):
    x = np.array(bits, dtype=int)

    original_objective = portfolio_objective(x, sigma=SIGMA_TEST)
    qubo_cost_value = float(x @ Q_test @ x + offset_test)

    # QUBO minimizes the negative of the original maximization objective
    error = abs(qubo_cost_value + original_objective)
    max_error = max(max_error, error)

print("Maximum QUBO equivalence error:", max_error)
print("QUBO equivalence verified:", np.isclose(max_error, 0.0))

_, best_feasible_check, _ = classical_baseline(RETURNS, SIGMA_TEST)
print("Best feasible classical objective:", best_feasible_check["objective"])
print("Best feasible asset indices:", best_feasible_check["selected_assets"])


Maximum QUBO equivalence error: 1.4210854715202004e-14
QUBO equivalence verified: True
Best feasible classical objective: 7.42
Best feasible asset indices: [0, 2]


In [33]:

experiment_results = {
    "experiment_type": "temporary_matrix_test",
    "risk_matrix": "SIGMA_TEST (not official)",
    "qaoa": {
        "p": QAOA_P,
        "best_gamma": float(best_gamma),
        "best_beta": float(best_beta),
        "optimizer_success": bool(optimization_result_2.success),
        "expected_qubo_cost": float(optimization_result_2.fun),
        "optimal_portfolio_probability": float(optimal_probability),
        "constraint_satisfaction_probability": float(valid_probability),
    },
    "classical": {
        "best_feasible_objective": float(best_feasible_check["objective"]),
        "best_feasible_assets": best_feasible_check["selected_assets"],
    },
    "processor_A": metrics_A,
    "processor_B": metrics_B,
    "limitations": [
        "Official risk matrix not yet supplied",
        "No hardware noise model used",
        "Routing overhead not yet isolated",
        "COBYLA did not report convergence",
    ],
}

print("Experiment results saved in memory.")
print(experiment_results)


Experiment results saved in memory.
{'experiment_type': 'temporary_matrix_test', 'risk_matrix': 'SIGMA_TEST (not official)', 'qaoa': {'p': 1, 'best_gamma': 1.1753022772507937, 'best_beta': 0.7887492867081691, 'optimizer_success': False, 'expected_qubo_cost': 1.7014347063758097, 'optimal_portfolio_probability': 0.29231294762166915, 'constraint_satisfaction_probability': 0.6368263318262894}, 'classical': {'best_feasible_objective': 7.42, 'best_feasible_assets': [0, 2]}, 'processor_A': {'depth': 91, 'physical_cx_count': 21, 'logical_cx_count': 12, 'additional_cx_gates': 9, 'cx_count_increase_percent': 75.0, 'explicit_swap_count': 0}, 'processor_B': {'depth': 85, 'physical_cx_count': 24, 'logical_cx_count': 12, 'additional_cx_gates': 12, 'cx_count_increase_percent': 100.0, 'explicit_swap_count': 0}, 'limitations': ['Official risk matrix not yet supplied', 'No hardware noise model used', 'Routing overhead not yet isolated', 'COBYLA did not report convergence']}


In [34]:

import json

results_file = "temporary_experiment_results.json"

with open(results_file, "w") as f:
    json.dump(experiment_results, f, indent=2)

print(f"Saved results to: {results_file}")

# Verify that the file can be read back
with open(results_file, "r") as f:
    loaded_results = json.load(f)

print("JSON verification:", loaded_results["experiment_type"])
print("Processors included:", "processor_A" in loaded_results and "processor_B" in loaded_results)


Saved results to: temporary_experiment_results.json
JSON verification: temporary_matrix_test
Processors included: True


## Experimental Results — Preliminary Test

### 1. Experimental Setup

A four-asset constrained portfolio optimization problem was formulated as a QUBO minimization problem. A depth-one QAOA circuit (\(p=1\)) was constructed and evaluated using an ideal statevector simulator. A temporary risk matrix, `SIGMA_TEST`, was used to validate the implementation because the official risk matrix was not available during this experiment.

### 2. Mathematical Verification

The QUBO representation was evaluated against all 16 possible bitstrings. The maximum absolute equivalence error was approximately \(1.42 \times 10^{-14}\), confirming numerical agreement between the QUBO cost and the negative of the original objective.

### 3. Preliminary Optimization Results

COBYLA returned the following best parameters in the test run:

- Gamma: 1.1753023
- Beta: 0.7887493
- Expected QUBO cost: 1.7014347
- Probability of measuring the optimal feasible portfolio: 29.23%
- Probability of selecting exactly two assets: 63.68%

The optimizer reached its evaluation limit and did not report successful convergence.

### 4. Processor Topology Comparison

| Metric | Processor A | Processor B |
|---|---:|---:|
| Physical qubits | 5 | 9 |
| Transpiled circuit depth | 91 | 85 |
| CNOT gates | 21 | 24 |
| RZ gates | 86 | 102 |
| SX gates | 40 | 48 |
| Explicit SWAP gates | 0 | 0 |

These measurements were obtained with the current Qiskit transpiler configuration, using optimization level 0 and seed 42. They are preliminary topology-only results, without hardware noise simulation.

### 5. Limitations and Next Steps

1. Replace `SIGMA_TEST` with the official risk matrix when it becomes available.
2. Rerun the classical baseline and QAOA optimization using the official inputs.
3. Establish a reproducible routing-overhead measurement.
4. Evaluate and report constraint satisfaction and objective quality using the official benchmark.
5. Export the updated notebook and results to the project GitHub repository.

**Important:** These results validate the implementation workflow but are not the final official benchmark results.

In [38]:

from google.colab import files

files.download("temporary_experiment_results.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>